# Import data

## Import player position/rotation data (savefile_01)
Read from .json and build dataframe

In [1]:
import json
import pandas as pd
import datetime as dt

FILENAME: str = "../data/savefile_01_pos_rot.json"
 
# 1. Load the raw JSON
with open(FILENAME, mode="r") as file:
    attempts_data: json = json.load(file)

# 2. Extract top-level metadata
player_id: str = attempts_data["playerId"]
save_time: dt.datetime = dt.datetime.strptime(attempts_data["saveDateTime"], "%Y-%m-%d %H:%M:%S")

# 3. Build rows from the list, reshaping position/rotation into tuples
dataframe_rows: list[dict] = []
for entry in attempts_data["playerSaveObjectList"]:
    dataframe_rows.append({
        "time":      entry["gametimer"],
        "position":  (entry["position"]["x"], entry["position"]["y"], entry["position"]["z"]),
        "rotation":  (entry["rotation"]["x"], entry["rotation"]["y"], entry["rotation"]["z"]),
        "playerId":  player_id,
        "saveTime":  save_time,
    })

# 4. Create the DataFrame
df: pd.DataFrame = pd.DataFrame(dataframe_rows)

x_list: list[float] =  [x for (x,y,z) in df["position"]]
z_list: list[float] =  [z for (x,y,z) in df["position"]]
rot_y_list: list[float] = [y for (x,y,z) in df["rotation"]]
t_list: list[float] = [_ for _ in df["time"]]

# print(df.head())
# print("-----------")
# print(df.dtypes)

## Import building data
Read from .json and import as Dataframe

In [2]:
import pandas as pd

BUILDING_FILENAME: str = "../data/buildings.json"
buildings_data: pd.DataFrame = pd.read_json(BUILDING_FILENAME).set_index("building_id")
#print(buildings_data)

## Import attemps data (savefile_02)


In [3]:
import pandas as pd
import json
import datetime as dt

ATTEMPTS_FILENAME: str = "../data/savefile_02_attempts.json"

# -------------------------- LOAD DATA --------------------------- #
# 1. Load JSON
with open(ATTEMPTS_FILENAME, mode="r") as file:
    attempts_data: json = json.load(file)

# 2. Extract top-level metadata
player_id: str = attempts_data["playerId"]
challenge_id: str  = attempts_data["trialSaveObjectList"][0]["challengeId"]
save_time: dt.datetime = dt.datetime.strptime(attempts_data["saveDateTime"], "%Y-%m-%d %H:%M:%S")

# 3. Build rows from the list
trial_rows: list[dict] = []
for entry in attempts_data["trialSaveObjectList"]:
    trial_rows.append({
         "time": float(entry["gameTimer"]),
         "challenge_id": entry["challengeId"],
         "attempts_count": entry["currentPhase"]["attemptsCount"],
         "target_building_id": entry["currentPhase"]["targetBuildingId"],
    })

trial_df: pd.DataFrame = pd.DataFrame(trial_rows)




# -------------------------- FUNCTIONS --------------------------- #
def get_attempt_num_for_time(time: float) -> int:
    """
    Returns the attempt number for a given play time.
    If there is no attempt number for the play time, returns '-1'
    """
    try:
        lowest_time_value_index: int = trial_df.loc[(trial_df.time < time)]["time"].idxmax()
    except ValueError:
        return -1
    
    return int(trial_df.loc[lowest_time_value_index]["attempts_count"]) #  without casting, returns np.int64(2)

def get_target_building_id_for_time(time: float) -> str:
    """
    Returns the attempt number for a given play time.
    If there is no attempt number for the play time, returns '-1'
    """
    try:
        lowest_time_value_index: int = trial_df.loc[(trial_df.time < time)]["time"].idxmax()

    except ValueError:
        return -1
    
    return trial_df.loc[lowest_time_value_index]["target_building_id"] #  casting not required for strings

# -------------------------- RUN --------------------------- #

print(trial_df)
# print(get_attempt_num_for_time(30))
# print(get_target_building_id_for_time(30))



        time challenge_id  attempts_count target_building_id
0  28.460131            1               2              L1-01
1  43.185093            1               3              L1-01
2  77.502213            1               4              L1-01


## Import attemps data (savefile_03)

### Cleanup
1. Replace challenge duration with latest value recorded
groupby().max() returns one row per group — it collapses the DataFrame. groupby().transform('max') broadcasts the result back to every row, keeping the original shape. So you can assign it back as a column without losing any rows.
```python
# Returns 1 row per challenge_id — can't assign back
df.groupby('challenge_id')['challenge_duration'].max()

# Returns a value for every original row — same length as df
df.groupby('challenge_id')['challenge_duration'].transform('max')
```

In [4]:
import json
import datetime as dt

ATTEMPTS_FILENAME: str = "../data/savefile_03_duration.json"

# -------------------------- LOAD DATA --------------------------- #

# 1. Load JSON
with open(ATTEMPTS_FILENAME, mode="r") as file:
    attempts_data: json = json.load(file)

# 2. Extract top-level metadata
player_id: str = attempts_data["playerId"]
challenge_id: str  = attempts_data["challengesList"][0]["challengeId"]
save_time: dt.datetime = dt.datetime.strptime(attempts_data["saveDateTime"], "%Y-%m-%d %H:%M:%S")

# 3. Build rows from the list
trial_rows: list[dict] = []
for entry in attempts_data["challengesList"]:
    trial_rows.append({
         "time": float(entry["gameTimer"]),
         "challenge_id": entry["challengeId"],
         "challenge_duration": entry["challengeDuration"],
         "attempt_number": entry["currentAttempt"]["attemptNumber"],
         "attempt_duration": entry["currentAttempt"]["attemptDuration"],
         "target_building_id": entry["currentAttempt"]["targetBuildingId"],
    })

trial_df: pd.DataFrame = pd.DataFrame(trial_rows)


# -------------------------- CLEAN UP DATA --------------------------- #

trial_df_clean = trial_df.copy()

# 1. Replace challenge duration with latest value recorded
trial_df_clean['challenge_duration'] = trial_df_clean.groupby('challenge_id')['challenge_duration'].transform('max')

# 2. Only keep the latest attempt information for each challenge
trial_df_clean = trial_df_clean.sort_values('time', ascending=True).drop_duplicates(subset=['challenge_id', 'attempt_number'], keep='last')
print("-----")
print(trial_df_clean)

# -------------------------- FUNCTIONS --------------------------- #
def get_attempt_num_for_time(time: float) -> int:
    """
    Returns the attempt number for a given play time.
    If there is no attempt number for the play time, returns '-1'
    """
    try:
        lowest_time_value_index: int = trial_df_clean.loc[(trial_df.time < time)]["time"].idxmax()
    except ValueError:
        return -1
    
    return int(trial_df_clean.loc[lowest_time_value_index]["attempt_number"]) #  without casting, returns np.int64(2)

def get_target_building_id_for_time(time: float) -> str:
    """
    Returns the attempt number for a given play time.
    If there is no attempt number for the play time, returns '-1'
    """
    try:
        lowest_time_value_index: int = trial_df.loc[(trial_df.time < time)]["time"].idxmax()

    except ValueError:
        return -1
    
    return trial_df.loc[lowest_time_value_index]["target_building_id"] #  casting not required for strings

# -------------------------- RUN --------------------------- #



print(get_attempt_num_for_time(20))
# print(get_target_building_id_for_time(30))




-----
         time challenge_id  challenge_duration  attempt_number  \
1   41.235168            1          117.889122               1   
3   73.968346            1          117.889122               2   
5  115.376244            1          117.889122               3   
7  125.298447            1          117.889122               4   

   attempt_duration target_building_id  
1         36.507854              L1-01  
3         30.721924              L1-01  
5         39.834061              L1-01  
7          8.172428              L1-01  
-1


# Build dataframe

In [5]:
# 3. Build rows from the list, reshaping position/rotation into tuples
dataframe_rows: list[dict] = []

for entry in attempts_data["playerSaveObjectList"]:
    time: float = float(entry["gameTimer"])
    dataframe_rows.append({
        "time": time,
        "attempt_num": get_attempt_num_for_time(time),
        "target_building_id": get_target_building_id_for_time(time),
        "position": (entry["position"]["x"], entry["position"]["y"], entry["position"]["z"]),
        "rotation": (entry["rotation"]["x"], entry["rotation"]["y"], entry["rotation"]["z"]),
        "playerId": player_id,
        "saveTime": save_time,
        "pos_x": entry["position"]["x"],
        "pos_z": entry["position"]["z"],
        "rot_y": entry["rotation"]["y"]
    })

# 4. Create the DataFrame
df: pd.DataFrame = pd.DataFrame(dataframe_rows)

x_list: list[float] =  [x for (x,y,z) in df["position"]]
z_list: list[float] =  [z for (x,y,z) in df["position"]]
rot_y_list: list[float] = [y for (x,y,z) in df["rotation"]]
t_list: list[float] = [_ for _ in df["time"]]


KeyError: 'playerSaveObjectList'

# Model with charts

## Map configuration for Level 1

In [ ]:
# MAP SIZE
GRID_COLS: int = 6
GRID_ROWS: int = 5
BLOCK_SIZE: int = 30

# The extent is computed — maps pixel corners to world coordinates
map_extent = [
    -BLOCK_SIZE / 2,
    (GRID_COLS - 0) * BLOCK_SIZE + BLOCK_SIZE / 2,
    -BLOCK_SIZE / 2,
    (GRID_ROWS - 0) * BLOCK_SIZE + BLOCK_SIZE / 2,
]

map_x_offset: int = BLOCK_SIZE / 2
map_z_offset: int = BLOCK_SIZE / 2

map_coord: list[int] = [
    -BLOCK_SIZE / 2,
    (GRID_COLS - 1) * BLOCK_SIZE + BLOCK_SIZE / 2,
    -BLOCK_SIZE / 2,
    (GRID_ROWS - 1) * BLOCK_SIZE + BLOCK_SIZE / 2,
]

map_coord[0] += map_x_offset  # left
map_coord[1] += map_x_offset  # right
map_coord[2] += map_z_offset  # bottom
map_coord[3] += map_z_offset  # top

## Map Graph

### Map Graph (plotly)

In [ ]:
import plotly.graph_objects as go
from PIL import Image
import numpy as np

# Load map image
map_img = Image.open("../images/map_level1.png")

SIZE = 4
base_triangle = np.array([
    [0, SIZE/2],
    [-SIZE/2, -SIZE/2],
    [SIZE/2, -SIZE/2],
])

# --- Build the figure with initial traces ---

fig = go.Figure()

# Trace 0: trail (static — never changes)
fig.add_trace(go.Scatter(
    x=x_list, y=z_list,
    mode='markers',
    marker=dict(color='white', size=5, opacity=0.25),
    name='Trail',
    showlegend=False
))

# Trace 1: buildings with labels (static — toggle via legend click)
fig.add_trace(go.Scatter(
    x=buildings_data["world_pos_x"].tolist(),
    y=buildings_data["world_pos_z"].tolist(),
    mode='markers+text',
    marker=dict(color='orange', size=12, symbol='square',
                line=dict(color='black', width=1)),
    text=buildings_data["name"].tolist(),
    textposition='top right',
    textfont=dict(size=9, color='black'),
    name='POIs'
))

# Trace 2: path (animated — starts empty)
fig.add_trace(go.Scatter(
    x=[], y=[],
    mode='lines+markers',
    marker=dict(color='cyan', size=4),
    line=dict(color='cyan', width=1),
    showlegend=False
))

# Trace 3: current position marker (animated — starts empty)
fig.add_trace(go.Scatter(
    x=[], y=[],
    mode='markers',
    marker=dict(color='red', size=12),
    showlegend=False
))

# Trace 4: rotation triangle (animated — starts empty)
fig.add_trace(go.Scatter(
    x=[], y=[],
    fill='toself',
    fillcolor='red',
    line=dict(color='white', width=1.5),
    showlegend=False
))

# --- Build all animation frames upfront ---

frames = []
for i in range(len(x_list)):
    # Rotation triangle
    next_frame = min(i + 1, len(x_list) - 1)
    angle = -rot_y_list[i]
    rad = np.radians(angle)
    cos_a, sin_a = np.cos(rad), np.sin(rad)
    rotated = np.array([
        [v[0]*cos_a - v[1]*sin_a, v[0]*sin_a + v[1]*cos_a]
        for v in base_triangle
    ])
    rotated[:, 0] += x_list[next_frame]
    rotated[:, 1] += z_list[next_frame]
    tri_x = list(rotated[:, 0]) + [rotated[0, 0]]  # close the shape
    tri_y = list(rotated[:, 1]) + [rotated[0, 1]]

    frames.append(go.Frame(
        data=[
            go.Scatter(x=x_list[:i+1], y=z_list[:i+1]),      # path
            go.Scatter(x=[x_list[i]], y=[z_list[i]]),          # current
            go.Scatter(x=tri_x, y=tri_y),                     # triangle
        ],
        traces=[2, 3, 4],  # which trace indices to update
        name=str(i)
    ))

fig.frames = frames

# --- Background image ---

fig.add_layout_image(
    source=map_img,
    xref="x", yref="y",
    x=map_coord[0],
    y=map_coord[3],       # plotly anchors images from top-left
    sizex=map_coord[1] - map_coord[0],
    sizey=map_coord[3] - map_coord[2],
    sizing="stretch",
    layer="below"
)

# --- Layout: axes, play button, slider ---

fig.update_layout(
    width=800, height=700,
    title="Player Position over Time",
    xaxis=dict(range=[map_extent[0], map_extent[1]], dtick=30, showgrid=True),
    yaxis=dict(range=[map_extent[2], map_extent[3]], dtick=30,
               showgrid=True, scaleanchor="x"),

    # Play/pause buttons
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        x=0.5, y=-0.05, xanchor="center",
        buttons=[
            dict(label="▶ Play", method="animate",
                 args=[None, dict(frame=dict(duration=100, redraw=True),
                                  fromcurrent=True)]),
            dict(label="⏸ Pause", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode="immediate")])
        ]
    )],

    # Time slider
    sliders=[dict(
        active=0,
        x=0.05, len=0.9,
        currentvalue=dict(prefix="Time: "),
        steps=[
            dict(
                args=[[str(i)], dict(frame=dict(duration=0, redraw=True),
                                      mode="immediate")],
                method="animate",
                label=f"{t_list[i]:.1f}s"
            )
            for i in range(len(x_list))
        ]
    )]
)

fig.show()

In [ ]:
import plotly.express as px

# Filter out -1
df_filtered = df[df["attempt_num"] != -1]

# Split tuples into columns
df_filtered["pos_x"] = df_filtered["position"].apply(lambda p: p[0])
df_filtered["pos_z"] = df_filtered["position"].apply(lambda p: p[2])

fig2 = px.line(
    df_filtered.sort_values("time"),        # ensure chronological order
    x="pos_x",
    y="pos_z",
    color="attempt_num",
    #hover_data=["time", "target_building_id"], # can add columns this, remove is manual
    hover_data={
      "pos_x": False,
      "pos_z": False,
    },
    labels={
      "attempt_num": "attempt number" 
    },
    markers=True,                   # dots + lines
    title="Player Path by Attempt"
)

# Lock axes so toggling attempts doesn't rescale
fig2.update_layout(
   xaxis=dict(range=[map_extent[0], map_extent[1]], dtick=30, showgrid=True),
   yaxis=dict(range=[map_extent[2], map_extent[3]], dtick=30,
              showgrid=True, scaleanchor="x"),
   width=800,
   height=700,
   updatemenus=[dict(
     type="buttons",
      showactive=True,
      x=1.005, y=.75, xanchor="left",
      buttons=[
         dict(
            label="Toggle Map",
            method="relayout",
            args=[{"images[0].visible": True}],
            args2=[{"images[0].visible": False}],  # second click
         )
      ]
   )]
)

# Background image (ready for overlay)
fig2.add_layout_image(
    source=map_img,
    xref="x", yref="y",
    x=map_coord[0],
    y=map_coord[3],
    sizex=map_coord[1] - map_coord[0],
    sizey=map_coord[3] - map_coord[2],
    sizing="stretch",
    layer="below"
)

fig2.add_annotation(
    text= "Challenge "+ challenge_id,
    x=1.15, y=1.15,
    xref="paper", yref="paper",
    showarrow=False,
    font=dict(size=14),
    name="challenge_label"
)

fig2.show()